**Прогнозирование цен на недвижимость (Boston Housing)**

В этом ноутбуке строится модель линейной регрессии для предсказания медианной стоимости жилья (`MedV`) по характеристикам района: уровень преступности, среднее число комнат, расстояние до центров занятости, налоги и т.д.

**Шаги:**
1. Загрузка датасета Boston.
2. Разделение на обучающую и тестовую выборки.
3. Обучение линейной регрессии.
4. Оценка качества (RMSE, MAE, R²).
5. Визуализация «предсказание vs. факт».

In [ ]:
import Pkg
Pkg.add([
    "MLJ",
    "MLJBase",
    "MLJModels",
    "MLJLinearModels",
    "DataFrames",
    "RDatasets",
    "Plots",
    "StatsBase",
    "Random"
])

In [ ]:
using MLJ
using RDatasets
using DataFrames
using Plots
using Random
using StatsBase

# Загружаем датасет Boston
boston = dataset("MASS", "Boston")
first(boston, 5)

In [ ]:
schema(boston)

In [ ]:
Random.seed!(42)

# 80% — обучение, 20% — тест
train, test = partition(boston, 0.8, shuffle=true, rng=42)

println("Обучающая выборка: $(nrow(train)) строк")
println("Тестовая выборка:  $(nrow(test)) строк")

In [ ]:
# Загружаем модель линейной регрессии из MLJLinearModels
LinearRegressor = @load LinearRegressor pkg=MLJLinearModels

linreg = LinearRegressor()

# Целевая переменная — MedV (медианная стоимость жилья)
X = select(train, Not(:MedV))
y = train.MedV

mach = machine(linreg, X, y)
fit!(mach)

println("Модель обучена.")

In [ ]:
X_test = select(test, Not(:MedV))
y_test = test.MedV

y_pred = predict(mach, X_test)

# Метрики качества
rmse_val = round(MLJ.rmse(y_pred, y_test), digits=3)
mae_val  = round(MLJ.mae(y_pred, y_test),  digits=3)
r2_val   = round(MLJ.r2(y_pred, y_test),   digits=3)

println("RMSE:  $rmse_val")
println("MAE:   $mae_val")
println("R²:    $r2_val")

In [ ]:
# Визуализация: предсказанные значения против реальных
scatter(y_test, y_pred,
    xlabel="Реальная цена (MedV)",
    ylabel="Предсказанная цена",
    title="Линейная регрессия: предсказание vs. факт",
    legend=false,
    alpha=0.6)

# Линия y = x (идеальное совпадение)
plot!(0:max(maximum(y_test), maximum(y_pred)),
      0:max(maximum(y_test), maximum(y_pred)),
      color=:red, lw=2, label="Идеал y = x")

In [ ]:
# Краткое резюме
println("=== Итог ===")
println("Размер обучающей выборки: $(nrow(train))")
println("Размер тестовой выборки:   $(nrow(test))")
println("RMSE = $rmse_val (тыс. долл.)")
println("MAE  = $mae_val  (тыс. долл.)")
println("R²   = $r2_val")
println("")
println("Модель объясняет примерно $(round(r2_val*100, digits=1))% дисперсии целевой переменной.")